# Imports de las librerías

In [77]:
# Imports de las librerías
import sqlite3
from bs4 import BeautifulSoup
import requests
import time
import re

# Conexión inicial a la BD (SQLite como DBMS)

In [78]:
# Conexión inicial a la BD
connection = sqlite3.connect("pengu_books.db")
cursor = connection.cursor()
print("Se realiza la conexión a la BD pengu_books")

Se realiza la conexión a la BD pengu_books


# Se crean las tablas
1) categories
2) authors
3) books
4) book_author


In [79]:
# FK activadas manualmente
cursor.execute("PRAGMA foreign_keys = ON")

# Tabla de categories
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT UNIQUE   NOT NULL
    )
""")

# Table de authors
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        name              TEXT    NOT NULL,
        birth_year        INTEGER,
        country           TEXT,
        external_api_id   TEXT    UNIQUE,
        total_known_works INTEGER,
        api_source        TEXT,
        created_at        DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# Tabla de books
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT    NOT NULL,
        price       REAL    NOT NULL CHECK (price >= 0),
        rating      INTEGER NOT NULL CHECK (rating >= 1 AND rating <= 5),
        category_id INTEGER NOT NULL,
        url         TEXT,
        created_at  DATETIME DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (category_id) REFERENCES categories(id)
    )
""")

# Tabla book_author (tabla puente N:M) 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id   INTEGER NOT NULL,
        author_id INTEGER NOT NULL,
        PRIMARY KEY (book_id, author_id),
        FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
        FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
    )
""")

connection.commit()
connection.close()
print("Base de datos creada: pengu_books.db")

Base de datos creada: pengu_books.db


# Reconexión a la BD

In [80]:
# Conexión a la BD
connection = sqlite3.connect("pengu_books.db")
cursor = connection.cursor()
print("Se realiza la reconexión a la BD pengu_books")

Se realiza la reconexión a la BD pengu_books


# Checking de la página "Books to Scrape"

In [81]:
# Checking rápido si responde la página de Books to Scrape

try:
    # Indicamos la URL 
    url_books_to_scrape = "https://books.toscrape.com/"

    # Se guarda la respuesta del método GET en la variable response
    url_response = requests.get(url_books_to_scrape)

    # Verificación del estado de conexión
    if url_response.status_code == 200:
        print("Conexión exitosa. La página Books to Scrape responde")

except (Exception,KeyboardInterrupt) as error:
    print(f"Hubo un error {error}")

Conexión exitosa. La página Books to Scrape responde


# Variables Globales


In [82]:
URL = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) PenguinAcademyCH/4.0"}

# Extracción de las categorías

In [ ]:
# Se inicia extrayendo inicialmente las categorías de los libros
response = requests.get(URL)
sopa = BeautifulSoup(response.text, "html.parser")

# Capturar la lista del panel lateral
lista_categorias = sopa.find("ul", class_="nav nav-list").find("li").find_all("li")

categorias_data = []
for categoria in lista_categorias:          
    nombre = categoria.text.strip()
    link = URL + categoria.a["href"]
    categorias_data.append((nombre, link))

cursor.executemany("INSERT OR IGNORE INTO categories (name) VALUES (?)", [(c[0],) for c in categorias_data])
connection.commit()

# Caché de mapeo id ↔ nombre
cursor.execute("SELECT id, name FROM categories")
categorias_db = {row[1]: row[0] for row in cursor.fetchall()}

print(f"{len(categorias_db)} categorías guardadas")


50 categorías guardadas


# Extracción de libros

In [50]:
rating = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
libros = []

def scrappear_categoria(nombre_categoria, url_categoria, id_categoria):
    libros_scrappeados = []
    url_actual = url_categoria

    while url_actual:
        response = requests.get(url_actual)
        sopa_categoria = BeautifulSoup(response.text, "html.parser")
        articulos = sopa_categoria.find_all("article", class_="product_pod")

        for articulo in articulos:                                              
            titulo = articulo.h3.a["title"].strip()
            precio = articulo.find("p", class_="price_color").text
            precio = float(precio.replace("£", "").replace("Â", "").strip())
            class_rating = articulo.find("p", class_="star-rating")["class"][1]
            estrellas = rating.get(class_rating, 0)

            link_relativo = articulo.h3.a["href"].replace("../../../", "")      
            url_libro = f"https://books.toscrape.com/catalogue/{link_relativo}"

            libros_scrappeados.append((titulo, precio, estrellas, id_categoria, url_libro))  

        boton_siguiente = sopa_categoria.find("li", class_="next")              
        if boton_siguiente:
            url_actual = url_actual.rsplit("/", 1)[0] + "/" + boton_siguiente.a["href"]
        else:
            url_actual = None

    return libros_scrappeados

print("Extrayendo las categorías")
tiempo_inicial = time.time()

for nombre, url in categorias_data:
    id_categoria = categorias_db[nombre]
    libros_de_cada_categoria = scrappear_categoria(nombre, url, id_categoria)
    libros.extend(libros_de_cada_categoria)

cursor.executemany("""
    INSERT OR IGNORE INTO books (title, price, rating, category_id, url)
    VALUES (?, ?, ?, ?, ?)
""", libros)
connection.commit()

# ❌ Borrar esto
# cursor.executemany("""
#     INSERT INTO books (title, price, rating, category_id, url)
#     VALUES (?, ?, ?, ?, ?)
# """, libros)

tiempo_final = time.time()                                                      
print(f"{len(libros)} libros extraídos")
print(f"El tiempo de inserción fue de {round(tiempo_final - tiempo_inicial)} segundos")

Extrayendo las categorías
1000 libros extraídos
El tiempo de inserción fue de 61 segundos


# Testeo de la API

In [76]:

titulo_ejemplo = "Hai to Gensou no Grimgar, Vol. 01 (Hai to Gensou no Grimgar #1)"

# recorte y limpieza del nombre (sin los paréntesis)
titulo_ejemplo_limpio = re.sub(r'\(.*?\)', '', titulo_ejemplo).strip()
print(f"Obra: {titulo_ejemplo}")
print(f"Título largo: {titulo_ejemplo}")
print(f"Título limpio:{titulo_ejemplo_limpio}\n")

# Simulando una búsqueda con datos inexistentes
print("--- Conexión a open library ---")
url_busqueda_prueba = "https://openlibrary.org/search.json"
parametros = {
    "title": titulo_ejemplo_limpio, 
    "limit": 1, 
    "fields": "title,author_name,author_key"
}

tiempo_prueba_inicial = time.time()

try:
    respuesta_ol = requests.get(url_busqueda_prueba, params=parametros, timeout=10)
    datos_ol = respuesta_ol.json()
    
    if datos_ol.get("docs"):
        doc = datos_ol["docs"][0]
        nombre_autor = doc.get("author_name", ["desconocido"])[0]
        id_autor = doc.get("author_key",  ["sin_id"])[0]
        
        print(f"autor encontrado: {nombre_autor}")
        print(f"id externo api:   {id_autor}")
        print("\njson devuelto por open library (fragmento):")
        print(doc)
    else:
        print("La Api no encontró el libro, incluso con el nombre recortado y limpio.")
        
except Exception as e:
    print(f"error en la búsqueda: {e}")

tiempo_prueba_final = time.time()

# Para tener un estimado de cuánto podría tardar con volúmenes altos
print(f"\nTiempo de consulta: {round(tiempo_prueba_final - tiempo_prueba_inicial, 3)} segundos")

Obra: Hai to Gensou no Grimgar, Vol. 01 (Hai to Gensou no Grimgar #1)
Título largo: Hai to Gensou no Grimgar, Vol. 01 (Hai to Gensou no Grimgar #1)
Título limpio:Hai to Gensou no Grimgar, Vol. 01

--- Conexión a open library ---
La Api no encontró el libro, incluso con el nombre recortado y limpio.

Tiempo de consulta: 4.151 segundos


# Enriquecimiento vía API

In [ ]:
# Bloque de enriquecimiento de datos vía APIs externas

# Mapeo de los países con la estructura de un diccionario clave|valor respondiendo a la estructura: gentilicio|país
mapeo_paises = {
    "american": "USA", 
    "british": "UK", 
    "english": "UK", 
    "scottish": "UK",
    "welsh": "UK", 
    "irish": "Ireland", 
    "french": "France", 
    "spanish": "Spain",
    "german": "Germany", 
    "italian": "Italy", 
    "canadian": "Canada",
    "australian": "Australia", 
    "japanese": "Japan", 
    "russian": "Russia",
    "chinese": "China", 
    "mexican": "Mexico", 
    "argentine": "Argentina",
    "colombian": "Colombia", 
    "chilean": "Chile", 
    "brazilian": "Brazil",
    "indian": "India", 
    "swedish": "Sweden", 
    "norwegian": "Norway",
    "dutch": "Netherlands"
}

# Se crea un caché para que quede un registro aquí y no se deba estar consultando repetidas veces si existe el autor.
cache_autores = {}

# Búsqueda mediante la API de open library
def buscar_en_openlibrary(book_id, titulo_original):
    titulo_limpio = re.sub(r'\(.*?\)', '', titulo_original).strip() # --> Aplicando lo que se hizo en el bloque de testo de la API

    # se completa la estructura nomas de los datos a ser rellenados. Solo la estructura
    datos = {
        "name": "Unknown", 
        "birth_year": None, 
        "country": None,
        "external_api_id": None, 
        "total_known_works": None,
        "api_source": "not_found"
    }

    try:
        # Intentos con el título ya recortado/limpio.
        params = {"title": titulo_limpio, "limit": 1, "fields": "title,author_name,author_key"}
        response = requests.get("https://openlibrary.org/search.json", params=params, headers=HEADERS, timeout=10)
        if not response.ok or not response.text.strip():
            return (book_id, datos)

        json_busqueda = response.json()

        # intento 2: sin subtitulo si falló el anterior
        if not json_busqueda.get("docs") and ":" in titulo_limpio:
            params["title"] = titulo_limpio.split(":")[0].strip()
            response = requests.get("https://openlibrary.org/search.json", params=params, headers=HEADERS, timeout=10)
            if response.ok and response.text.strip():
                json_busqueda = response.json()

        if not json_busqueda.get("docs"):
            return (book_id, datos)

        # Verificación de author_key y el author_name
        doc = json_busqueda["docs"][0]
        author_key  = doc.get("author_key",  [None])[0]
        author_name = doc.get("author_name", [None])[0]

        if not author_key or not author_name:
            return (book_id, datos)

        # Uso del caché, si ya se proceso este autor, reutilizar para ahorrar tiempo
        if author_key in cache_autores:
            return (book_id, cache_autores[author_key])

        datos["name"]            = author_name
        datos["external_api_id"] = author_key
        datos["api_source"]      = "open_library+wikipedia"

        # Detalles del autor
        clean_key = author_key.replace("/authors/", "")
        res_autor = requests.get(f"https://openlibrary.org/authors/{clean_key}.json", headers=HEADERS, timeout=10)

        # Fecha de nacimiento del autor
        if res_autor.status_code == 200 and res_autor.text.strip():
            birth_raw = res_autor.json().get("birth_date", "")
            if birth_raw:
                digitos = [s for s in birth_raw.split() if s.isdigit() and len(s) == 4]
                datos["birth_year"] = int(digitos[0]) if digitos else None


        # Consulta sobre las demás obras registradas
        res_obras = requests.get(f"https://openlibrary.org/authors/{clean_key}/works.json?limit=0", headers=HEADERS, timeout=10)
        
        if res_obras.status_code == 200 and res_obras.text.strip():
            datos["total_known_works"] = res_obras.json().get("size", None)

        # Se registra el resultado exitoso en la caché global
        cache_autores[author_key] = datos
        return (book_id, datos)

    except Exception as e:
        datos["api_source"] = f"error_ol: {str(e)[:20]}"
        return (book_id, datos)


# guardar todos los libros para su procesamiento
cursor.execute("SELECT id, title FROM books")
libros_db = cursor.fetchall()
resultados_ol = []

print("Enriquecimiento desde la API externa de open library")
t1 = time.time()

# Bucle de procesamiento por libro y por autor
for porcentaje, libro in enumerate(libros_db, 1):
    resultado = buscar_en_openlibrary(libro[0], libro[1])
    resultados_ol.append(resultado)
    if porcentaje % 25 == 0:
        print(f"  {porcentaje}/1000 libros procesados...")

print(f"Open library completado en {round(time.time()-t1, 2)}s")
encontrados = sum(1 for _, d in resultados_ol if d["external_api_id"])
print(f"  autores encontrados: {encontrados}/1000")


# Enriquecimiento vía Wikipedia

print("\n Enriquecimiento vía API Wikipedia")
t2 = time.time()

# Es solo para los registros que tienen id oficial de open library y nombre real
# almacenamiento de los datos de autores que no se repiten para las consultas de pais
autores_unicos = {}
for book_id, datos in resultados_ol:
    key = datos.get("external_api_id")
    if key and key not in autores_unicos and datos["name"] != "Unknown":
        autores_unicos[key] = datos

print(f"  autores unicos a consultar en wikipedia: {len(autores_unicos)}")

#consultar los autores en wikipedia para obtener sus paises
for i, (author_key, datos) in enumerate(autores_unicos.items()):
    if i % 50 == 0:
        print(f"  {i}/{len(autores_unicos)} autores procesados...")

    try:
        # rest_v1 y quote para evitar errores de codificacion en las url
        # quote es para utilizar formato de web en espavios
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(datos['name'])}"
        res = requests.get(url, headers=HEADERS, timeout=10)

        if res.status_code == 200 and res.text.strip():
            wiki_json = res.json()
            
            # extraer descripcion para buscar si hace referencia a un pais
            texto = (wiki_json.get("description", "") + " " + wiki_json.get("extract", "")).lower()

            for gentilicio, pais in mapeo_paises.items():
                if re.search(rf'\b{gentilicio}\b', texto):
                    datos["country"] = pais
                    # actualizacion de la memoria
                    cache_autores[author_key]["country"] = pais
                    break

    except Exception:
        pass

    # respeto al rate limit de la api
    time.sleep(0.8)  

print(f"Wikipedia completado en {round(time.time()-t2, 2)}s")


# Inyeccion de datos en la BD 
print("\n Añadiendo los registros en SQLite")

for book_id, info in resultados_ol:
    # Inserción del autor (sin tener en cuenta ya el id)
    cursor.execute("""
        INSERT OR IGNORE INTO authors
            (name, birth_year, country, external_api_id, total_known_works, api_source)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (info["name"], info["birth_year"], info["country"],
          info["external_api_id"], info["total_known_works"], info["api_source"]))

    # Recuperar el id local generado para la relación del libro
    if info["external_api_id"]:
        cursor.execute("SELECT id FROM authors WHERE external_api_id = ?", (info["external_api_id"],))
    else:
        cursor.execute("SELECT id FROM authors WHERE name = ?", (info["name"],))

    fila = cursor.fetchone()
    if fila:
        cursor.execute("INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)", (book_id, fila[0]))

connection.commit()

# Overview de los datos
cursor.execute("SELECT COUNT(*) FROM authors")
total = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM authors WHERE country IS NOT NULL")
con_pais = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM authors WHERE external_api_id IS NULL")
no_encontrados = cursor.fetchone()[0]

print(f"\n Pipeline completo: ")
print(f"  total de autores:              {total}")
print(f"  con pais:                   {con_pais}")
print(f"  no encontrados en API's:      {no_encontrados}")
print(f"  % de no encontrados:           {no_encontrados/len(libros_db)*100:.1f}%")